In [ ]:
# ------------------------------------------------------------
# Information Visualisation Coursework
# System B: Temporal Exploration of Road Traffic Accidents
#
# Dataset:
# Leeds Road Traffic Accidents (Kaggle)
# https://www.kaggle.com/datasets/thedevastator/leeds-road-traffic-accidents-2009-2018
#
# Description:
# This notebook implements System B, a time-centric interactive dashboard
# for exploring road traffic accidents in Leeds, featuring both generalised
# selection and bidirectional linking.
#
# Visualisation Tasks (Munzner, Ch.3 -- Actions + Targets):
#
#   T1 -- Explore spatial features  [Search -> Explore; Target: Feature]
#         Browse the accident map to discover geographic clusters and hotspot
#         areas where accidents concentrate across Leeds.
#         -> Map responds to time-period and hour selections; shows all
#            accidents when nothing is selected.
#
#   T2 -- Discover temporal distribution  [Analyse -> Discover; Target: Distribution]
#         Discover how accident frequency is distributed across hours of the day
#         to identify peak and low-risk time periods.
#         -> Hour bar chart is the primary T2 view, filtered by the selected
#            time-of-day period and the spatial map brush.
#
#   T3 -- Select a subset and summarise severity  [Search -> Browse; Query -> Summarise;
#         Target: Distribution]
#         Select a temporal subset (by period or specific hours) and/or a
#         spatial subset (by map brush), then summarise the severity profile.
#         *** This task involves selecting a subset of the data. ***
#         -> Period click OR hour brush OR map brush each define a subset;
#            the severity chart summarises the doubly/triply filtered result.
#
# Generalised Selection (Munzner -- semantic/hierarchical traversal):
#
#   Hierarchy:  Hour (level 1, specific) belongs to Time-of-Day Period (level 2, general)
#               Night        : hours  0-5
#               Morning Rush : hours  6-9
#               Daytime      : hours 10-15
#               Evening Rush : hours 16-19
#               Late Evening : hours 20-23
#
#   Traversal policy:
#               Clicking a time-of-day period bar GENERALISES the selection to
#               ALL hours within that semantic period.  The hour chart then shows
#               only hours belonging to the chosen period; the map and severity
#               chart are filtered accordingly.  Within the selected period the
#               user can NARROW back down by dragging the hour brush to pick
#               specific hours.  This is generalised selection -- not global
#               filtering -- because the expansion follows a pre-defined semantic
#               hierarchy (Morning Rush, Evening Rush, etc.) rather than an
#               arbitrary user-drawn range.
#
# Bidirectional Linking:
#   time_brush (hour interval) -- hour chart produces, map + severity consume
#   map_brush  (spatial interval) -- map produces, hour chart + severity consume
#   Selecting a region on the map reshapes the hour distribution; brushing hours
#   reshapes the spatial pattern -- both update simultaneously.
#
# Design Decisions (distinguishing B from A and C):
#   D1. Primary view:       A = spatial scatter map
#                           B = time-of-day period bar + hour bar
#                           C = condition stacked bar
#
#   D2. Selection mechanism: A = spatial interval brush + temporal interval brush
#                            B = time-period point click + temporal interval brush
#                                + spatial interval brush
#                            C = categorical point click + temporal interval brush
#                                + spatial interval brush
#
#   D3. Colour in primary:  A = severity (Slight/Serious/Fatal) on the map
#                           B = time-of-day period category on the period bar
#                           C = lighting condition (Daylight/Darkness) on condition bar
#
#   D4. Role of the map:    A = PRIMARY selection surface
#                           B = BIDIRECTIONAL node -- responds to time/period
#                               selections AND its spatial brush feeds back to
#                               the hour chart
#                           C = BIDIRECTIONAL hub -- responds to two independent
#                               inputs and its brush updates both condition bar
#                               and hour chart
#
#   D5. Severity chart:     A = filtered by spatial region AND hour range
#                           B = filtered by period AND time window AND spatial region
#                           C = filtered by road surface condition AND time window
#
#   D6. Linking direction:  A = spatial <-> temporal (bidirectional pair)
#                           B = hierarchical cascade with spatial feedback:
#                               period -> time -> space, plus space -> time
#                           C = dual bidirectional: condition <-> space and
#                               time <-> space
#
# Libraries used:
# pandas -- data processing
# altair -- interactive visualisation
# ------------------------------------------------------------

In [ ]:
# !pip install altair pandas

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

df = pd.read_csv("data/data.csv")
df = df[df["has_coords"] == True].copy()
df = df.dropna(subset=["hour", "severity"])
df["hour"]     = df["hour"].astype(int)
df["severity"] = df["severity"].astype(str)

print(df.shape)
df.head()

In [ ]:
# ---- Generalised selection: derive time-of-day period -------------------
# This defines the semantic hierarchy used for generalised selection.
# Each hour belongs to exactly one named period (the generalised level).

def hour_to_period(h):
    if   0 <= h <=  5: return "Night (0-5)"
    elif 6 <= h <=  9: return "Morning Rush (6-9)"
    elif 10 <= h <= 15: return "Daytime (10-15)"
    elif 16 <= h <= 19: return "Evening Rush (16-19)"
    else:               return "Late Evening (20-23)"

df["time_period"] = df["hour"].apply(hour_to_period)

PERIOD_ORDER = [
    "Night (0-5)",
    "Morning Rush (6-9)",
    "Daytime (10-15)",
    "Evening Rush (16-19)",
    "Late Evening (20-23)"
]

PERIOD_COLOURS = [
    "#2C3E50",   # Night        - dark navy
    "#E67E22",   # Morning Rush - orange
    "#F1C40F",   # Daytime      - yellow
    "#E74C3C",   # Evening Rush - red
    "#8E44AD"    # Late Evening - purple
]

print("Period distribution:")
print(df.groupby("time_period")["accident_id"].nunique().reindex(PERIOD_ORDER))

In [ ]:
# ---- Selections ---------------------------------------------------------

# GENERALISED selection: click a time-of-day period -> selects all hours in it
# empty=True: when nothing clicked, all data shows
period_sel = alt.selection_point(
    name="period_sel",
    fields=["time_period"],
    empty=True
)

# SPECIFIC selection: drag to pick a narrow hour range within the period
time_brush = alt.selection_interval(
    name="time_brush",
    encodings=["x"]
)

# BIDIRECTIONAL spatial selection: brush on map -> feeds back to hour chart
map_brush = alt.selection_interval(name="map_brush")

# Severity filter: click legend to highlight a severity on the map
severity_sel = alt.selection_point(
    name="severity_sel",
    fields=["severity"],
    bind="legend"
)

In [ ]:
# ---- Generalised Selection View: Time-of-Day Period Arc (Donut) ----------
# Clicking a segment generalises the selection to ALL hours in that period.
# Traversal: Hour (level 1) -> Time Period (level 2).
#
# Arc replaces the flat bar -- the proportional wedge areas immediately
# communicate which periods dominate accident counts at a glance.
# period_sel (selection_point on time_period field) works identically
# with mark_arc as it did with mark_bar.

period_bar = (
    alt.Chart(df,
        title="Generalised Selection: click a period to select all its hours")
    .mark_arc(innerRadius=60, outerRadius=130)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["time_period"]
    )
    .encode(
        theta=alt.Theta("accident:Q", stack=True),
        color=alt.Color(
            "time_period:N",
            sort=PERIOD_ORDER,
            scale=alt.Scale(domain=PERIOD_ORDER, range=PERIOD_COLOURS),
            legend=alt.Legend(title="Time Period")
        ),
        opacity=alt.condition(period_sel, alt.value(1.0), alt.value(0.25)),
        tooltip=[
            alt.Tooltip("time_period:N", title="Period"),
            alt.Tooltip("accident:Q",    title="Accidents")
        ]
    )
    .add_params(period_sel)
    .properties(width=280, height=280)
)

In [ ]:
# ---- T2: Hour Bar Chart (bidirectional, filtered by period + map) --------
# Consumes: period_sel (generalised) + map_brush (bidirectional spatial)
# Produces: time_brush (specific hour selection)
# When period is selected: only hours in that period are shown.
# When map is brushed: only hours for accidents in that region are shown.
# Uses distinct(accident_id) to count accidents not casualty rows.

hour_chart = (
    alt.Chart(df, title="T2 — Hour Distribution (drag to narrow within period)")
    .mark_bar()
    .transform_filter(period_sel)
    .transform_filter(map_brush)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["hour"]
    )
    .encode(
        x=alt.X("hour:O", title="Hour of Day (0-23)"),
        y=alt.Y("accident:Q", title="Number of Accidents"),
        color=alt.condition(
            time_brush,
            alt.value("#1f77b4"),
            alt.value("#d3d3d3")
        ),
        tooltip=[
            alt.Tooltip("hour:O",     title="Hour"),
            alt.Tooltip("accident:Q", title="Accidents")
        ]
    )
    .add_params(time_brush)
    .properties(width=420, height=280)
)

In [ ]:
# ---- T1: Map (bidirectional, filtered by period + time + severity) -------
# Consumes: period_sel + time_brush + severity_sel
# Produces: map_brush (spatial selection -> feeds back to hour chart)
# D4: Map is a BIDIRECTIONAL node in System B.

xmin, xmax = df["easting"].min(),  df["easting"].max()
ymin, ymax = df["northing"].min(), df["northing"].max()

map_chart = (
    alt.Chart(df,
        title="T1 — Accident Locations (drag map to feed back to hour chart)")
    .mark_circle(size=10, opacity=0.4)
    .transform_filter(period_sel)
    .transform_filter(time_brush)
    .transform_filter(severity_sel)
    .encode(
        x=alt.X(
            "easting:Q",
            title="Grid Ref Easting",
            scale=alt.Scale(domain=[xmin, xmax])
        ),
        y=alt.Y(
            "northing:Q",
            title="Grid Ref Northing",
            scale=alt.Scale(domain=[ymin, ymax])
        ),
        color=alt.Color("severity:N", title="Severity"),
        tooltip=[
            alt.Tooltip("accident_date:T", title="Date", format="%Y-%m-%d"),
            alt.Tooltip("hour:O",          title="Hour"),
            alt.Tooltip("time_period:N",   title="Period"),
            alt.Tooltip("severity:N",      title="Severity")
        ]
    )
    .add_params(map_brush)           # produces spatial selection (bidirectional)
    .properties(width=400, height=300)
)

In [ ]:
# ---- T3: Severity Breakdown (filtered by period + time + map) -----------
# Consumes: period_sel + time_brush + map_brush
# Produces: severity_sel (filters map by severity via legend click)
# D5: Severity chart filtered by period AND time window AND spatial region
#     -- unique to System B.
# Uses distinct(accident_id) to count accidents not casualty rows.
# Color consistent with System A and C (green / orange / red scale).

severity_chart = (
    alt.Chart(df, title="T3 — Severity of Selected Subset")
    .mark_bar()
    .transform_filter(period_sel)
    .transform_filter(time_brush)
    .transform_filter(map_brush)
    .transform_aggregate(
        accident="distinct(accident_id)",
        groupby=["severity"]
    )
    .encode(
        x=alt.X("severity:N", title="Severity"),
        y=alt.Y("accident:Q", title="Number of Accidents"),
        color=alt.condition(
            severity_sel,
            alt.Color(
                "severity:N",
                scale=alt.Scale(
                    domain=["Slight", "Serious", "Fatal"],
                    range=["#4CAF50", "#FC8D59", "#D73027"]
                ),
                title="Severity"
            ),
            alt.value("lightgray")
        ),
        tooltip=[
            alt.Tooltip("severity:N", title="Severity"),
            alt.Tooltip("accident:Q", title="Accidents")
        ]
    )
    .add_params(severity_sel)
    .properties(width=180, height=300)
)

In [ ]:
# ---- Dashboard Layout ---------------------------------------------------
#
# +---------------------------+------------------------------------------+
# |  Period arc (generalised  |  Hour bar (T2 -- drag to narrow;        |
# |  selection -- click wedge)|  filtered by period + map)              |
# +---------------------------+------------------+-----------------------+
# |  Spatial Map (T1)                           | Severity (T3)          |
# +---------------------------------------------+-----------------------+
#
# D6: Linking direction is period -> time -> space, with space feeding
#     back to time (bidirectional map <-> hour chart).

system_b = alt.vconcat(
    alt.hconcat(period_bar, hour_chart).resolve_scale(color="independent"),
    alt.hconcat(map_chart, severity_chart).resolve_scale(color="independent")
).configure_title(fontSize=13, anchor="start")

system_b